<a href="https://colab.research.google.com/github/JJingLu/CBS5055-Generative-Artificial-Intelligence-for-Innovative-Communications/blob/main/Social_Media_Data_Crawling_and_Analysis_Brand_%26_Audience_Insights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Workshop: Social Media Data Crawling and Analysis LST5414

Instructor: Jessie Lu

Welcome to Workshop!

In today’s session, you will learn how to build an end‑to‑end pipeline for collecting and analyzing social media data around a specific topic. We will start by using Python in Google Colab to scrape posts from platforms such as Weibo, including how to handle authentication, cookies, and pagination to reliably retrieve text content and metadata. You will then learn how to clean and structure the data with Pandas, export it to CSV, and compute basic engagement metrics such as likes, comments, and reposts/shares.

In [ ]:
# Step 1: Import necessary libraries for data collection and processing
import requests # Library for sending HTTP requests
import time # Library for time-related tasks
import random # Library for generating random numbers
import re # Library for regular expression operations
from urllib.parse import unquote # Function to decode URL-encoded strings
import pandas as pd # Library for data manipulation and analysis
import datetime # Library for date and time handling
import os # Library for operating system dependent functionality
import sys # Library for system-specific parameters and functions
from google.colab import files # Module for downloading files from Colab

## Instructions for obtaining your Weibo cookie


1.   Log in to Weibo on your own browser\
Open https://m.weibo.cn/  in Chrome/Edge and log in with your personal account. Make sure you can see the homepage or the search results page (e.g. for the keyword “AI”).
2.   Open the browser Developer Tools
*   On Windows: press F12 or Ctrl + Shift + I
*   On macOS: press ⌥ + ⌘ + I (Option + Command + I)

A new panel will appear, usually docked to the right or bottom of the window.\

3.   Go to the Network tab and reload the page\
In Developer Tools, click the “Network” tab.
Check “Preserve log” if available.
Press Ctrl + R / ⌘ + R to reload the page so that network requests start to appear.
4.   Filter and select an API request that returns data\
In the filter box, type getIndex (this is the Weibo mobile API used in our code).
Click on one of the requests like api/container/getIndex?....
In the right panel, confirm under “Preview” or “Response” that you see JSON data (fields like data, cards, mblog, etc.), not an HTML error page.

5.   Copy the Cookie header from that request\
Still inside this request, go to the “Headers” tab.
Scroll down to “Request Headers” and find the line starting with Cookie:.
Right‑click on the value and choose “Copy value” (or manually select and copy everything after Cookie:).
Paste this full string into your notebook/code where the variable Cookie or cookie_str is defined (replacing the placeholder).

In [ ]:
# Step 2: Define the scraping logic and data cleaning functions

def clean_weibo_html(raw_html):
    """
    Strips various HTML tags (e.g., emojis, hyperlinks, line breaks) from Weibo text using regex
    to maintain neutrality and readability.
    """
    if not raw_html:
        return ""
    # Replace break tags with actual newlines
    text = re.sub(r'<br\s*/?>', '\n', raw_html)
    # Strip all other HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    return text.strip()

def scrape_weibo_search(keyword="AI", target_count=200):
    # 1. Base interface definition for the Weibo mobile API
    api_url = "https://m.weibo.cn/api/container/getIndex"

    # 2. Construct search parameters based on the keyword
    containerid = f"100103type=1&q={keyword}"

    # 3. Define request headers with authentication cookies
    headers = {
        "Accept": "application/json, text/plain, */*",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6",
        # NOTE: Use your captured browser Cookie here
        "Cookie": "_T_WM=85826531316; WEIBOCN_FROM=1110006030; XSRF-TOKEN=6d1b1b; MLOGIN=1; M_WEIBOCN_PARAMS=luicode%3D10000011%26lfid%3D100103type%253D1%2526q%253DAI%26fid%3D100103type%253D1%2526q%253DAI%26uicode%3D10000011; SUB=_2A25Eq3NpDeRhGeRG6VER-C3Myz-IHXVnyYqhrDV6PUJbktANLRj4kW1NTc_nmk3R7Sd0d0BBAi5kk1j4TT0yh-rf;",
        "Priority": "u=1, i",
        "Referer": f"https://m.weibo.cn/search?containerid={containerid}",
        "Sec-Ch-Ua": '"Not(A:Brand";v="24", "Chromium";v="122", "Microsoft Edge";v="122"',
        "Sec-Ch-Ua-Mobile": "?1",
        "Sec-Ch-Ua-Platform": '"Android"',
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Mobile Safari/537.36 Edg/122.0.0.0",
        "X-Requested-With": "XMLHttpRequest",
        "X-Xsrf-Token": "6d1b1b"
    }

    collected_data = []
    page = 1
    consecutive_errors = 0

    print(f"Initializing crawler: Target keyword '{keyword}', expected to fetch {target_count} items...")

    # 4. Start the pagination loop to fetch data until the target count is reached
    while len(collected_data) < target_count:
        params = {
            "containerid": containerid,
            "page_type": "searchall",
            "page": page
        }

        try:
            # Send the GET request to the API
            response = requests.get(api_url, headers=headers, params=params, timeout=15)

            # Check if the request was successful
            if response.status_code != 200:
                print(f"[Blocked] Request failed, HTTP {response.status_code}. Risk control triggered or token expired.")
                break

            data = response.json()

            # Validate business status code
            if data.get("ok") != 1:
                print(f"[Warning] Abnormal business status (ok={data.get('ok')}). No more data or authentication failed.")
                break

            cards = data.get("data", {}).get("cards", [])
            if not cards:
                print("[Finished] cards array is empty, reached end of search results.")
                break

            extracted_in_page = 0

            # 5. Extract post information from the JSON response
            for card in cards:
                card_group = card.get("card_group", [])
                if not card_group and card.get("card_type") == 9:
                    card_group = [card]

                for item in card_group:
                    if item.get("card_type") == 9:
                        mblog = item.get("mblog", {})
                        raw_html_text = mblog.get("text", "")
                        clean_text = clean_weibo_html(raw_html_text)

                        # Store structured information
                        post_info = {
                            "author": mblog.get("user", {}).get("screen_name", "Unknown"),
                            "created_at": mblog.get("created_at", ""),
                            "text": clean_text,
                            "reposts_count": mblog.get("reposts_count", 0),
                            "comments_count": mblog.get("comments_count", 0),
                            "attitudes_count": mblog.get("attitudes_count", 0)
                        }

                        collected_data.append(post_info)
                        extracted_in_page += 1

                        if len(collected_data) >= target_count:
                            break

                if len(collected_data) >= target_count:
                    break

            print(f"Page {page} completed: extracted {extracted_in_page} posts. Current total: {len(collected_data)} / {target_count}")

            page += 1
            consecutive_errors = 0

            # 6. Anti-crawling measure: Random sleep interval
            sleep_time = random.uniform(4.5, 8.2)
            time.sleep(sleep_time)

        except requests.exceptions.RequestException as e:
            print(f"[Network Error] Request layer error: {e}")
            consecutive_errors += 1
        except ValueError:
            print(f"[Parsing Error] JSON decoding failed, response might be an HTML error page.")
            consecutive_errors += 1

        if consecutive_errors >= 3:
            print("[Forced Termination] 3 consecutive fatal errors, exiting to protect IP safety.")
            break

    # 7. Finalize results: convert to DataFrame, save to CSV, and trigger download
    print(f"Task execution completed. Actual data count: {len(collected_data)}")

    if not collected_data:
        print("No data acquired, CSV not generated.")
        return collected_data

    df = pd.DataFrame(collected_data)

    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"weibo_{keyword}_{len(collected_data)}_{ts}.csv"
    filepath = os.path.join("/content", filename)

    df.to_csv(filepath, index=False, encoding="utf-8-sig")
    print(f"Saved to Colab: {filepath}")

    files.download(filepath)

    return collected_data

# Execution logic
if __name__ == "__main__":
    results = scrape_weibo_search(keyword="AI", target_count=200)

    for idx, post in enumerate(results[:3], 1):
        print(f"\n--- Sample {idx} ---")
        print(f"Author: {post['author']}")
        print(f"Time: {post['created_at']}")
        print(f"Text: {post['text'][:100]}...")

def save_data_to_disk(data_list, keyword="AI"):
    """
    Persists scraped in-memory data to local disk in CSV and Excel formats.
    """
    if not data_list:
        print("[Interrupt] No data in memory to save.")
        return

    try:
        df = pd.DataFrame(data_list)

        columns_mapping = {
            "author": "Post Author",
            "created_at": "Post Time",
            "text": "Weibo Text",
            "reposts_count": "Reposts",
            "comments_count": "Comments",
            "attitudes_count": "Likes"
        }
        df.rename(columns=columns_mapping, inplace=True)

        current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        base_filename = f"weibo_{keyword}_{current_time}"
        csv_filename = f"{base_filename}.csv"
        excel_filename = f"{base_filename}.xlsx"

        print(f"Writing {len(df)} records to CSV structure...")
        df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"[Success] CSV file saved to current directory: {csv_filename}")

        try:
            print(f"Writing {len(df)} records to Excel structure...")
            df.to_excel(excel_filename, index=False, engine='openpyxl')
            print(f"[Success] Excel file saved to current directory: {excel_filename}")
        except ModuleNotFoundError:
            print("[Warning] openpyxl engine missing, unable to save as .xlsx.")
        except Exception as excel_err:
            print(f"[Exception] Failed to save Excel file: {excel_err}")

    except Exception as e:
        print(f"[Fatal Error] Data persistence process crashed: {e}", file=sys.stderr)

## Data Preparation for Analysis

### Subtask:
Convert the collected Weibo data (from the `results` variable) into a Pandas DataFrame and preprocess necessary columns like 'created_at' for temporal analysis. This step will also handle potential empty data scenarios gracefully.


In [ ]:
# Step 3: Data Preparation and Temporal Feature Engineering

if not results:
    # Handle cases where no data was retrieved to prevent downstream errors
    print("No data available in 'results' to process.")
else:
    # 1. Convert the list of dictionaries into a structured Pandas DataFrame
    df_weibo = pd.DataFrame(results)

    # 2. Convert 'created_at' strings to datetime objects for analysis
    # The format follows: 'Mon Mar 09 12:19:11 +0800 2026'
    df_weibo['created_at'] = pd.to_datetime(df_weibo['created_at'], format='%a %b %d %H:%M:%S %z %Y')

    # 3. Extract temporal components to identify posting patterns
    df_weibo['year'] = df_weibo['created_at'].dt.year
    df_weibo['month'] = df_weibo['created_at'].dt.month
    df_weibo['day'] = df_weibo['created_at'].dt.day
    df_weibo['hour'] = df_weibo['created_at'].dt.hour
    df_weibo['day_of_week'] = df_weibo['created_at'].dt.day_name()

    print("DataFrame 'df_weibo' created and temporal features extracted.")
    print(df_weibo.head())

## Engagement Metrics Analysis (Analysis 1 & Visualization)

### Subtask:
Calculate and display average engagement metrics (reposts, comments, likes) and visualize them using a bar chart for easy comparison.


In [ ]:
# Step 4: Engagement Metrics Analysis
import matplotlib.pyplot as plt

# 1. Calculate average metrics
avg_reposts = df_weibo['reposts_count'].mean()
avg_comments = df_weibo['comments_count'].mean()
avg_likes = df_weibo['attitudes_count'].mean()

engagement_summary = {
    'Average Reposts': avg_reposts,
    'Average Comments': avg_comments,
    'Average Likes': avg_likes
}

# 2. Display text results
print("Average Engagement Metrics Summary:")
for metric, value in engagement_summary.items():
    print(f"{metric}: {value:.2f}")

# 3. Visualization: Average Engagement Metrics Bar Chart
labels = list(engagement_summary.keys())
values = list(engagement_summary.values())

plt.figure(figsize=(10, 6))
plt.bar(labels, values, color=['#3498db', '#e74c3c', '#2ecc71'])
plt.title('Average Engagement Metrics for Collected Weibo Posts')
plt.ylabel('Average Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Temporal Posting Patterns (Analysis 2)

### Subtask:
Analyze and visualize the posting frequency over different time granularities (e.g., hour of day, day of week, day of month) to identify patterns.


In [ ]:
# Step 5: Temporal Posting Patterns Analysis
import matplotlib.pyplot as plt

# 1. Posting frequency by hour of the day to identify peak activity times
hourly_frequency = df_weibo.groupby('hour').size().reset_index(name='post_count')

plt.figure(figsize=(12, 6))
plt.bar(hourly_frequency['hour'], hourly_frequency['post_count'], color='skyblue')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Posts')
plt.title('Posting Frequency by Hour of Day')
plt.xticks(range(0, 24))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 2. Posting frequency by day of the week to observe weekly trends
day_of_week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_frequency = df_weibo.groupby('day_of_week').size().reindex(day_of_week_order).reset_index(name='post_count')

plt.figure(figsize=(10, 6))
plt.bar(daily_frequency['day_of_week'], daily_frequency['post_count'], color='lightcoral')
plt.xlabel('Day of Week')
plt.ylabel('Number of Posts')
plt.title('Posting Frequency by Day of Week')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 3. Posting frequency by day of the month to see long-term patterns
day_of_month_frequency = df_weibo.groupby('day').size().reset_index(name='post_count')

plt.figure(figsize=(12, 6))
plt.bar(day_of_month_frequency['day'], day_of_month_frequency['post_count'], color='lightgreen')
plt.xlabel('Day of Month')
plt.ylabel('Number of Posts')
plt.title('Posting Frequency by Day of Month')
plt.xticks(range(1, max(day_of_month_frequency['day']) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Simple Sentiment Analysis (Analysis 3)

### Subtask:
Perform a basic sentiment analysis on the Weibo text content to determine the positivity score. This will involve installing a suitable Chinese NLP library if not already present and applying it to the text data.


In [ ]:
!pip install snownlp
print("snownlp installed successfully.")

In [ ]:
# Step 6: Sentiment Analysis using SnowNLP
from snownlp import SnowNLP

# 1. Define a function to retrieve sentiment scores (0 to 1 range)
def get_sentiment(text):
    if pd.isna(text) or not text.strip():
        return None
    try:
        return SnowNLP(text).sentiments
    except Exception as e:
        print(f"Error processing text for sentiment: {e}")
        return None

# 2. Apply the function to create a new sentiment score column
df_weibo['sentiment_score'] = df_weibo['text'].apply(get_sentiment)

print("DataFrame with sentiment scores:")
print(df_weibo.head())

In [ ]:
import matplotlib.pyplot as plt

# Visualize the distribution of sentiment scores
plt.figure(figsize=(10, 6))
plt.hist(df_weibo['sentiment_score'].dropna(), bins=20, color='purple', alpha=0.7)
plt.xlabel('Sentiment Score (0=Negative, 1=Positive)')
plt.ylabel('Number of Posts')
plt.title('Distribution of Weibo Sentiment Scores')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Display descriptive statistics for sentiment scores
print("\nDescriptive statistics for Sentiment Scores:")
print(df_weibo['sentiment_score'].describe())

## Correlation Analysis (Analysis 4)

### Subtask:
Calculate the correlation between the sentiment (positivity score) of the Weibo content and the number of likes, and visualize this relationship using a scatter plot.


In [ ]:
# Step 7: Correlation and Relationship Analysis
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate the Pearson correlation between sentiment and likes
correlation = df_weibo['sentiment_score'].corr(df_weibo['attitudes_count'])
print(f"Pearson correlation: {correlation:.2f}")

# 2. Visualize the relationship using a scatter plot with a regression trend
plt.figure(figsize=(10, 6))
sns.scatterplot(x='sentiment_score', y='attitudes_count', data=df_weibo, alpha=0.6)
plt.title('Sentiment Score vs. Attitudes Count (Likes)')
plt.xlabel('Sentiment Score (0=Negative, 1=Positive)')
plt.ylabel('Attitudes Count')
plt.grid(True, linestyle='--', alpha=0.7)

# 3. Annotate the plot with the correlation coefficient
plt.text(0.05, 0.95, f'Correlation: {correlation:.2f}', transform=plt.gca().transAxes,
         fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', fc='yellow', alpha=0.5))
plt.show()

## Step 8: Latent Dirichlet Allocation (LDA) Preprocessing

### Subtask:
Install necessary libraries (jieba, gensim), define a list of Chinese stop words, and tokenize the Weibo text data to prepare it for topic modeling.


In [ ]:
!pip install jieba gensim
print("Libraries jieba and gensim installed.")

In [ ]:
import re
import jieba

# Step 8: Latent Dirichlet Allocation (LDA) Preprocessing

# Recommended: Prepare a stopwords.txt (one word per line)
# Fallback to a small set of common internal stop words if no file exists
DEFAULT_STOPWORDS = {
    "的", "了", "在", "是", "我", "你", "他", "她", "它", "我们", "你们", "他们",
    "这", "那", "和", "与", "及", "或", "也", "都", "就", "还", "而", "但", "因为", "所以",
    "一个", "没有", "什么", "这样", "那样", "自己", "可以", "可能", "觉得", "不是",
}

# Regex for handling punctuation, symbols, and whitespace (Chinese/English)
PUNCT_RE = re.compile(r"^[\s\W_]+$", flags=re.UNICODE)

def load_stopwords(path: str | None = None, extra: set[str] | None = None):
    stopwords = set(DEFAULT_STOPWORDS)

    if path:
        try:
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    w = line.strip()
                    if w:
                        stopwords.add(w)
        except FileNotFoundError:
            print(f"[Warning] stopwords file not found: {path}. Using DEFAULT_STOPWORDS only.")

    if extra:
        stopwords |= set(extra)

    return stopwords

# Decide whether to treat 'AI' as a stop word
STOPWORDS = load_stopwords(
    path=None,                  # e.g., "stopwords.txt"
    extra={"ai", "AI"}           # Remove this line if you want to keep 'AI'
)

In [ ]:
def preprocess_chinese_text(text, stopwords=STOPWORDS, min_len=2):
    if not isinstance(text, str) or not text.strip():
        return []

    # Normalization: strip whitespace (add more cleaning here if needed)
    text = text.strip()

    # jieba word segmentation
    words = jieba.lcut(text, cut_all=False)

    cleaned = []
    for w in words:
        w = w.strip()
        if not w:
            continue

        # Filter out pure punctuation/symbols/whitespace
        if PUNCT_RE.match(w):
            continue

        # Case normalization and stop word filtering (prevents missing AI/ai)
        if w.lower() in {s.lower() for s in stopwords}:
            continue

        # Filter short words (Standard LDA practice: remove single characters)
        if len(w) < min_len:
            continue

        cleaned.append(w)

    return cleaned

# Apply to df_weibo
df_weibo["tokenized_text"] = df_weibo["text"].apply(preprocess_chinese_text)

print("Tokenization complete. Inspecting 'tokenized_text' column:")
print(df_weibo[["text", "tokenized_text"]].head())

In [ ]:
from gensim.corpora import Dictionary

texts = df_weibo["tokenized_text"].tolist()

# Create the dictionary
dictionary = Dictionary(texts)

# Filter extremes (adjust based on data volume)
dictionary.filter_extremes(
    no_below=5,     # Keep tokens that appear in at least 5 documents
    no_above=0.5,   # Keep tokens that appear in no more than 50% of documents
    keep_n=5000     # Keep only the first 5000 most frequent tokens
)

# Create the corpus: Bag-of-Words representation
corpus = [dictionary.doc2bow(t) for t in texts]

print("Number of documents:", len(texts))
print("Vocabulary size:", len(dictionary))
print("Example BOW:", corpus[0][:10])

In [ ]:
from gensim.models import LdaModel

NUM_TOPICS = 8
lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=10,
    iterations=200,
    alpha="auto",
    eta="auto"
)

# Print the top words for each topic
for i in range(NUM_TOPICS):
    print(f"\nTopic {i}:")
    print(lda.print_topic(i, topn=10))

In [ ]:
def get_dominant_topic(bow, lda_model):
    topics = lda_model.get_document_topics(bow, minimum_probability=0)
    dominant_topic, dominant_prob = max(topics, key=lambda x: x[1])
    return dominant_topic, dominant_prob

dominant = [get_dominant_topic(bow, lda) for bow in corpus]
df_weibo["dominant_topic"] = [t for t, p in dominant]
df_weibo["topic_prob"] = [p for t, p in dominant]

df_weibo[["text", "dominant_topic", "topic_prob"]].head()

In [ ]:
!pip install pyLDAvis
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Prepare the visualization data
vis_data = gensimvis.prepare(lda, corpus, dictionary)
# Display the interactive topic model visualization
pyLDAvis.display(vis_data)